In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Draw
from collections import Counter
from PIL import Image, ImageDraw
import math
from random import choice
from rdkit.Chem.Draw import rdMolDraw2D
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from random import shuffle
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
sdf_file = "./external_data/2D_structures.sdf"

In [ ]:
suppl = Chem.SDMolSupplier(sdf_file, removeHs=True)
mols = [m for m in suppl if m is not None]

In [ ]:
len(set(mols))

In [ ]:

# Extract Bemis–Murcko scaffolds
def get_scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold.GetNumAtoms() == 0:
        return None
    smi = Chem.MolToSmiles(scaffold, canonical=True, isomericSmiles=False)
    return Chem.MolToSmiles(Chem.MolFromSmiles(smi), canonical=True, isomericSmiles=False)

scaffolds = [get_scaffold_smiles(m) for m in mols]
nscaff = [s for s in scaffolds if s is None]
scaffolds = [s for s in scaffolds if s is not None]


In [ ]:
print(len(set(scaffolds)))

In [ ]:
from collections import Counter
c = Counter(scaffolds)
print(c['O=C1CC(c2ccccc2)Oc2ccccc21'])   
print(c['O=c1cc(-c2ccccc2)oc2ccccc12'])

# Scaffold analysis

In [ ]:
aquachem = pd.read_csv('../Data/MAIN_chemical_structure_data.tsv', sep='\t', dtype=str)
aquachem = aquachem.fillna('')
aquachem['pubchemid'] = aquachem['pubchemid'].str.replace('CID:', '')
print(aquachem.shape)
aquachem.head()

In [ ]:
aqdf = aquachem[aquachem['pubchemid']!='']
aqdf = aqdf[['pubchemid', 'id']]
print(aqdf.shape)

In [ ]:
scaffold_data = []
for m in mols:
    smiles = get_scaffold_smiles(m)
    if smiles is not None:
        cid = m.GetProp("PUBCHEM_COMPOUND_CID") if m.HasProp("PUBCHEM_COMPOUND_CID") else None
        scaffold_data.append({"pubchemid": cid, "scaffold": smiles})

In [ ]:
scaffold_df = pd.DataFrame(scaffold_data)
print(scaffold_df.shape)
scaffold_df.head()

In [ ]:
scaffold_df[scaffold_df['scaffold'].isin(['O=C1CC(c2ccccc2)Oc2ccccc21', 'O=c1cc(-c2ccccc2)oc2ccccc12'])]

In [ ]:
scaffold_df['scaffold'].value_counts()

In [ ]:
scaffold_df = scaffold_df.merge(aqdf, on='pubchemid', how='left')
print(scaffold_df.shape)
scaffold_df.head()

#### Regulation

In [ ]:
regs = pd.read_csv('./external_data/combined_regulations.tsv', sep='\t', dtype=str)
print(regs.shape)
regs.head()

In [ ]:
regs = regs[['ID', 'Regulatory Status']].drop_duplicates().rename(columns={'ID':'id', 'Regulatory Status':'status'})
print(regs.shape)
regs.head()

In [ ]:
regs['status'].value_counts()

In [ ]:
regs = regs.groupby(['id']).agg(lambda x: '|'.join(set(sorted(x.to_list())))).reset_index()
print(regs.shape)
regs.head()

In [ ]:
regs['modified_status'] = regs['status']
regs = regs[['id', 'modified_status']].drop_duplicates()
print(regs.shape)
regs.head()

In [ ]:
len(set(regs['id']))

In [ ]:
regs['modified_status'].value_counts()

In [ ]:
scaffold_df1 = scaffold_df.merge(regs, on='id', how='left')
scaffold_df1 = scaffold_df1.fillna('')
print(scaffold_df1.shape)
scaffold_df1.head()

In [ ]:
scaffold_df1['modified_status'].value_counts()

In [ ]:
len(set(scaffold_df1[scaffold_df1['modified_status']!='']['id']))

In [ ]:
scaffold_df1.columns

In [ ]:
df_exploded = (
    scaffold_df1[scaffold_df1["modified_status"] != ""]
    .assign(modified_status=lambda x: x["modified_status"].str.split("|"))
    .explode("modified_status")
)
df_exploded["modified_status"] = df_exploded["modified_status"].str.strip()

In [ ]:
print(len(set(df_exploded['scaffold'])))
print(len(set(df_exploded['id'])))
print(len(set(df_exploded['modified_status'])))
df_exploded['modified_status'].value_counts()

# Figures

In [ ]:
# Count scaffold frequencies
scaffold_counts = Counter(scaffolds)
print(len(set(scaffold_counts)))

# keep only frequent scaffolds
scaffold_counts = {s: c for s, c in scaffold_counts.items() if c >= 2}

In [ ]:
print(len(scaffold_counts))
scaffold_counts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from rdkit import Chem
from rdkit.Chem import Draw
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


# Setup

CMAP = plt.cm.YlOrRd

scaffolds = scaffold_df1["scaffold"].tolist()
scaffold_counts = Counter(scaffolds)
scaffold_counts = {s: c for s, c in scaffold_counts.items() if c >= 2}

df = scaffold_df1[scaffold_df1["scaffold"].isin(scaffold_counts)].copy()
df = df[df["modified_status"].str.strip() != ""]


# Explode multi-status rows

df["statuses"] = df["modified_status"].str.split("|").apply(
    lambda x: [s.strip() for s in x]
)
df_exploded = df.explode("statuses")


# Build presence matrix

scaffold_order = (
    df_exploded.groupby("scaffold")["id"]
    .count()
    .sort_values(ascending=False)
    .head(5)
    .sort_values(ascending=True)
    .index.tolist()
)
status_order = ["Approved", "Banned", "Regulated", "Partially approved", "Not approved", "Registered"]

presence = pd.crosstab(df_exploded["scaffold"], df_exploded["statuses"])
presence = presence.reindex(index=scaffold_order, columns=status_order, fill_value=0)
status_order = [s for s in status_order if presence[s].sum() > 0]
presence = presence[status_order]


# Molecule images

def smiles_to_img(smi, size=(260, 140)):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.ones((size[1], size[0], 3), dtype=np.uint8) * 255
    return np.array(Draw.MolToImage(mol, size=size))


# Plot

n_scaffolds = len(scaffold_order)
n_statuses  = len(status_order)

img_col_w = 2.4
heat_col_w = max(3, n_statuses * 0.5)
fig_h = max(6, n_scaffolds * 0.85 + 2)

fig = plt.figure(figsize=(img_col_w + heat_col_w, fig_h))
gs  = fig.add_gridspec(1, 2, width_ratios=[img_col_w, heat_col_w], wspace=0.04)
ax_img  = fig.add_subplot(gs[0])
ax_heat = fig.add_subplot(gs[1])

# draw heatmap cells
max_count = presence.values.max()
norm = Normalize(vmin=0, vmax=max_count)

for yi, smi in enumerate(scaffold_order):
    for xi, status in enumerate(status_order):
        count = presence.loc[smi, status]
        color = CMAP(norm(count)) if count > 0 else "#F0F0F0"
        ax_heat.add_patch(
            plt.Rectangle(
                (xi - 0.45, yi - 0.38), 0.9, 0.76,
                color=color, linewidth=0
            )
        )
        if count > 0:
            ax_heat.text(
                xi, yi, str(count),
                ha="center", va="center",
                fontsize=8, color="black", fontweight="bold"
            )
        else:
            ax_heat.add_patch(
                plt.Rectangle(
                    (xi - 0.45, yi - 0.38), 0.9, 0.76,
                    color="#F0F0F0", linewidth=0
                )
            )

# grid lines
for xi in range(n_statuses + 1):
    ax_heat.axvline(xi - 0.5, color="white", linewidth=1.5)
for yi in range(n_scaffolds + 1):
    ax_heat.axhline(yi - 0.5, color="white", linewidth=1.5)

# axes formatting
ax_heat.set_xlim(-0.5, n_statuses - 0.5)
ax_heat.set_ylim(-0.5, n_scaffolds - 0.5)
ax_heat.set_xticks(range(n_statuses))
ax_heat.set_xticklabels(status_order, rotation=35, ha="right", fontsize=9)
ax_heat.set_yticks(range(n_scaffolds))
ax_heat.set_yticklabels([""] * n_scaffolds)
ax_heat.tick_params(axis="both", length=0)
ax_heat.set_title("Regulatory status by scaffold", fontsize=12,
                  fontweight="bold", pad=10)

# colorbar
sm = ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_heat, shrink=0.4, pad=0.02)
cbar.set_label("Number of compounds", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# molecule images
ax_img.set_xlim(0, 1)
ax_img.set_ylim(-0.5, n_scaffolds - 0.5)
ax_img.axis("off")
ax_img.set_title("Scaffold", fontsize=11, pad=8)

for i, smi in enumerate(scaffold_order):
    arr = smiles_to_img(smi)
    im  = OffsetImage(arr, zoom=0.42)
    ab  = AnnotationBbox(
        im, (0.5, i), frameon=False,
        bboxprops=dict(edgecolor="lightgrey", linewidth=0.6,
                       boxstyle="round,pad=0.1")
    )
    ax_img.add_artist(ab)

plt.savefig("./scaffold_regulation_heatmap.svg", dpi=300)
plt.show()

In [ ]:
scaffold_order

In [ ]:
df.groupby("scaffold")["id"].count()

#### Therapeutic effects

In [ ]:
ther_data = pd.read_csv('./Data/MAIN_curated_therapeutic_effects_data.tsv', sep='\t', dtype=str)
ther_data = ther_data.fillna('')
print(ther_data.shape)
ther_data.head()

In [ ]:
ther_data = ther_data[['id', 'therapeutic_action']].drop_duplicates()
print(ther_data.shape)
ther_data.head()

In [ ]:
scaffold_df2 = scaffold_df.merge(ther_data, on='id', how='left')
scaffold_df2 = scaffold_df2.fillna('')
print(scaffold_df2.shape)
scaffold_df2.head()

In [ ]:
scaffold_df2['therapeutic_action'].value_counts()

In [ ]:
len(set(scaffold_df2[scaffold_df2['therapeutic_action']!='']['id']))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from rdkit import Chem
from rdkit.Chem import Draw
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

CMAP = plt.cm.YlOrRd  



# Setup

PALETTE = {
    "Mortality":       "#DC3535",
    "Physiology":      "#1D9E75",
    "Enzyme(s)":       "#3366AD",
    "Biochemistry":    "#7F77DD",
    "Cell(s)":         "#D4537E",
    "Genetics":        "#E8810A",
    "Development":     "#4CB5AE",
    "Histology":       "#F0C419",
    "Behavior":        "#6AAF3D",
    "Injury":          "#B05CC8",
    "Growth":          "#8B4513",
    "Feeding behavior":"#2E86AB",
    "Accumulation":    "#E8705A",
    "Immunological":   "#C49A6C",
    "Morphology":      "#5C7A29",
}

scaffolds = scaffold_df2["scaffold"].tolist()
scaffold_counts = Counter(scaffolds)
scaffold_counts = {s: c for s, c in scaffold_counts.items() if c >= 2}

df = scaffold_df2[scaffold_df2["scaffold"].isin(scaffold_counts)].copy()
df = df[df["therapeutic_action"].str.strip() != ""]

df_exploded = df



scaffold_order = (
    df.groupby("scaffold")["id"]
    .count()
    .sort_values(ascending=False)  
    .head(5)                        
    .sort_values(ascending=True)    
    .index.tolist()
)

presence = pd.crosstab(df_exploded["scaffold"], df_exploded["therapeutic_action"])
status_order = [s for s in presence.columns if presence[s].sum() > 0]
presence = presence[status_order]


# Molecule images

def smiles_to_img(smi, size=(260, 140)):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.ones((size[1], size[0], 3), dtype=np.uint8) * 255
    return np.array(Draw.MolToImage(mol, size=size))


# Plot

n_scaffolds = len(scaffold_order)
n_statuses  = len(status_order)

img_col_w = 2.4
heat_col_w = max(3, n_statuses * 0.5)
fig_h = max(6.5, n_scaffolds * 1 + 2.7)

fig = plt.figure(figsize=(img_col_w + heat_col_w, fig_h))
gs  = fig.add_gridspec(1, 2, width_ratios=[img_col_w, heat_col_w], wspace=0.04)
ax_img  = fig.add_subplot(gs[0])
ax_heat = fig.add_subplot(gs[1])

# draw heatmap cells
max_count = presence.values.max()
norm = Normalize(vmin=0, vmax=max_count)

for yi, smi in enumerate(scaffold_order):
    for xi, status in enumerate(status_order):
        count = presence.loc[smi, status]
        color = CMAP(norm(count)) if count > 0 else "#F0F0F0"
        ax_heat.add_patch(
            plt.Rectangle(
                (xi - 0.45, yi - 0.38), 0.9, 0.76,
                color=color, linewidth=0
            )
        )
        if count > 0:
            ax_heat.text(
                xi, yi, str(count),
                ha="center", va="center",
                fontsize=8, color="black", fontweight="bold"
            )

# grid lines
for xi in range(n_statuses + 1):
    ax_heat.axvline(xi - 0.5, color="white", linewidth=1.5)
for yi in range(n_scaffolds + 1):
    ax_heat.axhline(yi - 0.5, color="white", linewidth=1.5)

# axes formatting
ax_heat.set_xlim(-0.5, n_statuses - 0.5)
ax_heat.set_ylim(-0.5, n_scaffolds - 0.5)
ax_heat.set_xticks(range(n_statuses))
ax_heat.set_xticklabels(status_order, rotation=35, ha="right", fontsize=9)
ax_heat.set_yticks(range(n_scaffolds))
ax_heat.set_yticklabels([""] * n_scaffolds)
ax_heat.tick_params(axis="both", length=0)
ax_heat.set_title("Therapeutic action by scaffold", fontsize=12,
                  fontweight="bold", pad=10)

# legend
sm = ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_heat, shrink=0.4, pad=0.02)
cbar.set_label("Number of compounds", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# molecule images
ax_img.set_xlim(0, 1)
ax_img.set_ylim(-0.5, n_scaffolds - 0.5)
ax_img.axis("off")
ax_img.set_title("Scaffold", fontsize=11, pad=8)

for i, smi in enumerate(scaffold_order):
    arr = smiles_to_img(smi)
    im  = OffsetImage(arr, zoom=0.42)
    ab  = AnnotationBbox(
        im, (0.5, i), frameon=False,
        bboxprops=dict(edgecolor="lightgrey", linewidth=0.6,
                       boxstyle="round,pad=0.1")
    )
    ax_img.add_artist(ab)

plt.savefig("./scaffold_ther_heatmap.svg", dpi=300)
plt.show()

In [ ]:
scaffold_order

#### Toxic effects

In [ ]:
tox_data = pd.read_csv('./Data/MAIN_curated_toxic_effects_data.tsv', sep='\t', dtype=str)
tox_data = tox_data.fillna('')
print(tox_data.shape)
tox_data.head()

In [ ]:
tox_data = tox_data[['id', 'effect']].drop_duplicates()
print(tox_data.shape)
tox_data.head()

In [ ]:
scaffold_df3 = scaffold_df.merge(tox_data, on='id', how='left')
scaffold_df3 = scaffold_df3.fillna('')
print(scaffold_df3.shape)
scaffold_df3.head()

In [ ]:
scaffold_df3['effect'].value_counts()

In [ ]:
len(set(scaffold_df3[scaffold_df3['effect']!='']['id']))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from rdkit import Chem
from rdkit.Chem import Draw
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

CMAP = plt.cm.YlOrRd  



# Setup

PALETTE = {
    "Mortality":       "#DC3535",
    "Physiology":      "#1D9E75",
    "Enzyme(s)":       "#3366AD",
    "Biochemistry":    "#7F77DD",
    "Cell(s)":         "#D4537E",
    "Genetics":        "#E8810A",
    "Development":     "#4CB5AE",
    "Histology":       "#F0C419",
    "Behavior":        "#6AAF3D",
    "Injury":          "#B05CC8",
    "Growth":          "#8B4513",
    "Feeding behavior":"#2E86AB",
    "Accumulation":    "#E8705A",
    "Immunological":   "#C49A6C",
    "Morphology":      "#5C7A29",
}

scaffolds = scaffold_df3["scaffold"].tolist()
scaffold_counts = Counter(scaffolds)
scaffold_counts = {s: c for s, c in scaffold_counts.items() if c >= 2}

df = scaffold_df3[scaffold_df3["scaffold"].isin(scaffold_counts)].copy()
df = df[df["effect"].str.strip() != ""]

df_exploded = df



scaffold_order = (
    df.groupby("scaffold")["id"]
    .count()
    .sort_values(ascending=False)  
    .head(5)                       
    .sort_values(ascending=True)   
    .index.tolist()
)
status_order = list(PALETTE.keys())

presence = pd.crosstab(df_exploded["scaffold"], df_exploded["effect"])
presence = presence.reindex(index=scaffold_order, columns=status_order, fill_value=0)
status_order = [s for s in presence.columns if presence[s].sum() > 0]
presence = presence[status_order]


# Molecule images

def smiles_to_img(smi, size=(260, 140)):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.ones((size[1], size[0], 3), dtype=np.uint8) * 255
    return np.array(Draw.MolToImage(mol, size=size))


# Plot

n_scaffolds = len(scaffold_order)
n_statuses  = len(status_order)

img_col_w = 2.4
heat_col_w = max(3, n_statuses * 0.5)
fig_h = max(6.5, n_scaffolds * 1 + 2.7)

fig = plt.figure(figsize=(img_col_w + heat_col_w, fig_h))
gs  = fig.add_gridspec(1, 2, width_ratios=[img_col_w, heat_col_w], wspace=0.04)
ax_img  = fig.add_subplot(gs[0])
ax_heat = fig.add_subplot(gs[1])

# draw heatmap cells
max_count = presence.values.max()
norm = Normalize(vmin=0, vmax=max_count)

for yi, smi in enumerate(scaffold_order):
    for xi, status in enumerate(status_order):
        count = presence.loc[smi, status]
        color = CMAP(norm(count)) if count > 0 else "#F0F0F0"
        ax_heat.add_patch(
            plt.Rectangle(
                (xi - 0.45, yi - 0.38), 0.9, 0.76,
                color=color, linewidth=0
            )
        )
        if count > 0:
            ax_heat.text(
                xi, yi, str(count),
                ha="center", va="center",
                fontsize=8, color="black", fontweight="bold"
            )

# grid lines
for xi in range(n_statuses + 1):
    ax_heat.axvline(xi - 0.5, color="white", linewidth=1.5)
for yi in range(n_scaffolds + 1):
    ax_heat.axhline(yi - 0.5, color="white", linewidth=1.5)

# axes formatting
ax_heat.set_xlim(-0.5, n_statuses - 0.5)
ax_heat.set_ylim(-0.5, n_scaffolds - 0.5)
ax_heat.set_xticks(range(n_statuses))
ax_heat.set_xticklabels(status_order, rotation=35, ha="right", fontsize=9)
ax_heat.set_yticks(range(n_scaffolds))
ax_heat.set_yticklabels([""] * n_scaffolds)
ax_heat.tick_params(axis="both", length=0)
ax_heat.set_title("Toxic effect by scaffold", fontsize=12,
                  fontweight="bold", pad=10)

# legend
sm = ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_heat, shrink=0.4, pad=0.02)
cbar.set_label("Number of compounds", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# molecule images
ax_img.set_xlim(0, 1)
ax_img.set_ylim(-0.5, n_scaffolds - 0.5)
ax_img.axis("off")
ax_img.set_title("Scaffold", fontsize=11, pad=8)

for i, smi in enumerate(scaffold_order):
    arr = smiles_to_img(smi)
    im  = OffsetImage(arr, zoom=0.42)
    ab  = AnnotationBbox(
        im, (0.5, i), frameon=False,
        bboxprops=dict(edgecolor="lightgrey", linewidth=0.6,
                       boxstyle="round,pad=0.1")
    )
    ax_img.add_artist(ab)

plt.savefig("./scaffold_toxicity_heatmap.svg", dpi=300)
plt.show()

In [ ]:
scaffold_order

### Scaffold cloud

In [ ]:

# Size scaling

def tile_size_from_count(count, min_size=120, max_size=420):
    return int(
        min_size
        + (max_size - min_size)
        * math.log(count + 1)
        / math.log(max(scaffold_counts.values()) + 1)
    )

In [ ]:

# Pastel colours

PASTEL_COLORS = [
    (255, 245, 235),
    (235, 245, 255),
    (245, 235, 255),
    (235, 255, 245),
    (255, 235, 245),
    (245, 255, 235),
]

In [ ]:

# Create a coloured scaffold tile

def scaffold_tile(smiles, tile_size, bg_color, count, border=2):
    mol = Chem.MolFromSmiles(smiles)

    # draw molecule with transparent background
    drawer = rdMolDraw2D.MolDraw2DCairo(
        tile_size - 20,
        tile_size - 20
    )
    drawer.drawOptions().clearBackground = False
    rdMolDraw2D.PrepareAndDrawMolecule(drawer, mol)
    drawer.FinishDrawing()

    mol_png = drawer.GetDrawingText()
    mol_img = Image.open(BytesIO(mol_png)).convert("RGBA")

    # background tile
    tile = Image.new("RGBA", (tile_size, tile_size), bg_color + (255,))
    draw = ImageDraw.Draw(tile)

    # border
    draw.rectangle(
        [(0, 0), (tile_size - 1, tile_size - 1)],
        outline=(150, 150, 150),
        width=border
    )

    # center molecule
    x = (tile_size - mol_img.width) // 2
    y = (tile_size - mol_img.height) // 2
    tile.paste(mol_img, (x, y), mol_img)

    # frequency label (top-right)
    # font (fallback-safe)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", size=max(14, tile_size // 10))
    except IOError:
        font = ImageFont.load_default()

    text = str(count)
    text_w, text_h = draw.textbbox((0, 0), text, font=font)[2:]

    pad = 5
    tx = tile_size - text_w - pad - 4
    ty = pad + 2

    # white backing for readability
    draw.rectangle(
        [
            (tx - pad, ty - pad),
            (tx + text_w + pad, ty + text_h + pad),
        ],
        fill=(255, 255, 255, 220)
    )

    draw.text(
        (tx, ty),
        text,
        fill=(80, 80, 80),
        font=font
    )

    return tile



In [ ]:

# Rectangle overlap test

def overlaps(box, others, pad=6):
    x1, y1, x2, y2 = box
    for ox1, oy1, ox2, oy2 in others:
        if not (
            x2 + pad < ox1 or
            x1 - pad > ox2 or
            y2 + pad < oy1 or
            y1 - pad > oy2
        ):
            return True
    return False



# Sort scaffolds by frequency

sorted_scaffolds = sorted(
    scaffold_counts.items(),
    key=lambda x: -x[1]
)

In [ ]:
# Detect neighbors
def is_neighbor(box1, box2, margin=0):
    x1, y1, x2, y2 = box1
    a1, b1, a2, b2 = box2

    return not (
        x2 + margin < a1 or
        x1 - margin > a2 or
        y2 + margin < b1 or
        y1 - margin > b2
    )


In [ ]:
def choose_non_neighbor_color(x, y, w, h, placed_boxes, palette):
    current_box = (x, y, x + w, y + h)

    margin = int(0.35 * max(w, h))

    neighbor_colors = set()

    for px1, py1, px2, py2, pcolor in placed_boxes:
        if is_neighbor(current_box, (px1, py1, px2, py2), margin=margin):
            neighbor_colors.add(pcolor)

    colors = palette[:]
    shuffle(colors)

    for color in colors:
        if color not in neighbor_colors:
            return color

    return min(
        palette,
        key=lambda c: sum(1 for *_, pc in placed_boxes if pc == c)
    )

Create and save

In [ ]:
# Create canvas
canvas_size = (2800, 2800)
canvas = Image.new("RGBA", canvas_size, (255, 255, 255, 255))
cx, cy = canvas_size[0] // 2, canvas_size[1] // 2

placed_boxes = []



# Place central (most frequent) scaffold
center_smi, center_count = sorted_scaffolds[0]
center_size = tile_size_from_count(center_count)
center_color = choice(PASTEL_COLORS)

center_tile = scaffold_tile(
    center_smi,
    center_size,
    center_color,
    center_count
)

x = cx - center_tile.width // 2
y = cy - center_tile.height // 2

canvas.paste(center_tile, (x, y))
placed_boxes.append(
    (x, y, x + center_tile.width, y + center_tile.height, center_color)
)


# Spiral placement

angle_step = 0.20
radius_step = 10
start_radius = center_tile.width // 2 + 30
max_radius = min(canvas_size) // 2 - 40

for smi, count in sorted_scaffolds[1:]:
    size = tile_size_from_count(count)

    angle = 0.0
    radius = start_radius
    placed_flag = False

    while radius < max_radius:
        x = int(cx + radius * math.cos(angle) - size / 2)
        y = int(cy + radius * math.sin(angle) - size / 2)

        color = choose_non_neighbor_color(
            x, y,
            size, size,
            placed_boxes,
            PASTEL_COLORS
        )

        geom_box = (x, y, x + size, y + size)

        if not overlaps(geom_box, [b[:4] for b in placed_boxes]):
            tile = scaffold_tile(smi, size, color, count)
            canvas.paste(tile, (x, y))
            placed_boxes.append((x, y, x + size, y + size, color))
            placed_flag = True
            break

        angle += angle_step
        radius += radius_step * angle_step / (2 * math.pi)

    if not placed_flag:
        print("Could not place one scaffold without overlap.")


In [ ]:
scaffold_counts #verify with the figure